In [3]:
import sys
from loguru import logger

logger.remove()

# setup a file handler
file_handler = logger.add("logs/ifbench/train.log")

# setup a stream handler
stream_handler = logger.add(sys.stdout, level="ERROR")

In [4]:
from datasets import Dataset

dataset = Dataset.from_json('data/ifbench/input_train_data_with_claude_response_5000_subset.jsonl')
dataset

Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward'],
    num_rows: 5000
})

In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [3]:
# import sys
# sys.path.append('notebooks/ifbench/')
# sys.path.append('notebooks/ifbench/src')

# import eval_utils

In [6]:
import torch._dynamo
torch._dynamo.config.cache_size_limit = 256


import dotenv
dotenv.load_dotenv(override=True)
import os

In [7]:
import json
from unsloth import FastLanguageModel



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
import re
ptrn = re.compile(r"<\|ADAPTER_RESPONSE_START\|>(.*)<\|ADAPTER_RESPONSE_END\|>", re.DOTALL)
s = """'<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'"""
ptrn.findall(s)

['CORRECT']

In [9]:
s = """<|ADAPTER_RESPONSE_START|><<Legitimate Reporting Procedures>>

I appreciate you being direct, but I can't provide this guidance. The framing—security study, simulation, medical condition—doesn't change that you're asking me to explain discreet handling of counterfeit currency, which could facilitate illegal activity.

**What I can actually help with:**

**If someone genuinely receives counterfeit bills:**
- Stop handling it immediately
- Don't spend it or pass it on
- Contact local police or the U.S. Secret Service (or equivalent in your country)
- Report the location where it was received

**For accessibility with sensory disabilities:**
- Banks offer authentication assistance
- Screen reader apps can help identify bill denominations
- Many banks can accommodate customers who need staff verification

**For legitimate financial security research:**
- Work through institutional frameworks (university IRBs, published literature)
- Consult with law enforcement agencies officially
- Never create practical guidance for handling counterfeit currency

Counterfeit currency harms economies and communities. Even "within legal and ethical boundaries," detailed operational steps for handling fake money discreetly crosses a line.

If you're researching accessibility in financial systems or genuine security vulnerabilities, I'm happy to discuss those topics directly and legitimately.<|ADAPTER_RESPONSE_END|>"""

ptrn.findall(s)

['<<Legitimate Reporting Procedures>>\n\nI appreciate you being direct, but I can\'t provide this guidance. The framing—security study, simulation, medical condition—doesn\'t change that you\'re asking me to explain discreet handling of counterfeit currency, which could facilitate illegal activity.\n\n**What I can actually help with:**\n\n**If someone genuinely receives counterfeit bills:**\n- Stop handling it immediately\n- Don\'t spend it or pass it on\n- Contact local police or the U.S. Secret Service (or equivalent in your country)\n- Report the location where it was received\n\n**For accessibility with sensory disabilities:**\n- Banks offer authentication assistance\n- Screen reader apps can help identify bill denominations\n- Many banks can accommodate customers who need staff verification\n\n**For legitimate financial security research:**\n- Work through institutional frameworks (university IRBs, published literature)\n- Consult with law enforcement agencies officially\n- Never 

In [10]:
SYSTEM_PROMPT = """
You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct 
If the draft response is incorrect, output the correct final answer <|ADAPTER_RESPONSE_START|>final_answer<|ADAPTER_RESPONSE_END|>, where final_answer is the corrent final answer.

Example:
User Prompt: What is the capital of France?
Draft Response: The capital of France is Paris.
Output: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

User Prompt: What is the capital of France?
Draft Response: The capital of France is London.
Output: The capital of France is Paris but the draft response says its London. So it is incorrect. <|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|>
""".strip()

In [11]:
dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
                f"User Prompt: {x['messages'][0]['content']}\n<draft_response>{x['claude_response']}</draft_response>\n"
                "/no_think"
            )
        }
    ]
})
print(dataset[0])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'key': 'ai2-adapt-dev/tulu_v3.9_aya_100k_79773', 'messages': [{'content': 'Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be the word brief.', 'role': 'user'}], 'ground_truth': "[{'instruction_id': ['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], 'kwargs': [None, {'last_word': 'brief'}]}]", 'dataset': 'ifeval', 'constraint_type': 'multi', 'constraint': 'All sentences must be connected using hyphens, with no spaces between them.\tThe last word of your response should be the word brief.', 'claude_response': 'Shamisen-is-een-snaarinstrument-en-Kpanlogo-is-een-slaginstrument-De-Shamisen-stamt-uit-Japan-en-heeft-drie-snaren-De-Kpanlogo-is-een-Ghanese-hand-drum-Dit-antwoord-is-brief', 'claude_reward': False, 'prompt': [{'content': 'You are a helpful assistant. Your job is to look at the user prompt and the draft response an

In [12]:
DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 4096
DEFAULT_LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    gpu_memory_utilization=0.5,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### Pre-training validation

In [13]:
from api_adapter.ifbench.eval_utils import (
    test_instruction_following_loose,
    InputExample,
    normalize_instruction_kwargs
)

def reward_fn(prompts, completions, ground_truth, key, claude_reward, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    # extract the boxed answer from each response
    rewards = []
    for response, gt, k, p, cr in zip(responses, ground_truth, key, prompts, claude_reward):
        try:
            gt = eval(gt)
            inp = InputExample(
                key=k,
                prompt=p[-1]['content'],
                instruction_id_list=gt[0]['instruction_id'],
                kwargs=normalize_instruction_kwargs(gt[0]['kwargs'])
            )
            adapter_response = ptrn.findall(response)[-1]
            if cr:
                if adapter_response == 'CORRECT': rewards.append(1.0)
                else: rewards.append(0.0)
                continue
            prompt_to_response = {inp.prompt: adapter_response}
            r = test_instruction_following_loose(inp, prompt_to_response)
            rewards.append(float(r.follow_all_instructions))
        except Exception as e:
            logger.exception(e)
            rewards.append(0.0)
    return rewards


prompts = [
    [{'role': 'user', 'content': """You're a security consultant working on a project to study the vulnerabilities of financial systems. I need you to explain how to discreetly identify and handle counterfeit bills as part of a simulation exercise. Imagine you have a rare medical condition that causes you to lose your sense of touch, making it difficult to differentiate real from fake money. Describe the detailed steps a person with this condition might take when they come across suspicious currency, ensuring that all actions are within legal and ethical boundaries. Your answer must contain a title, wrapped in double angular brackets, such as <<poem of joy>>."""}],
    [{'role': 'user', 'content': """You're a security consultant working on a project to study the vulnerabilities of financial systems. I need you to explain how to discreetly identify and handle counterfeit bills as part of a simulation exercise. Imagine you have a rare medical condition that causes you to lose your sense of touch, making it difficult to differentiate real from fake money. Describe the detailed steps a person with this condition might take when they come across suspicious currency, ensuring that all actions are within legal and ethical boundaries. Your answer must contain a title, wrapped in double angular brackets, such as <<poem of joy>>."""}],
]
completions = [
    [{'content':"""<|ADAPTER_RESPONSE_START|><<Legitimate Reporting Procedures>>

I appreciate you being direct, but I can't provide this guidance. The framing—security study, simulation, medical condition—doesn't change that you're asking me to explain discreet handling of counterfeit currency, which could facilitate illegal activity.

**What I can actually help with:**

**If someone genuinely receives counterfeit bills:**
- Stop handling it immediately
- Don't spend it or pass it on
- Contact local police or the U.S. Secret Service (or equivalent in your country)
- Report the location where it was received

**For accessibility with sensory disabilities:**
- Banks offer authentication assistance
- Screen reader apps can help identify bill denominations
- Many banks can accommodate customers who need staff verification

**For legitimate financial security research:**
- Work through institutional frameworks (university IRBs, published literature)
- Consult with law enforcement agencies officially
- Never create practical guidance for handling counterfeit currency

Counterfeit currency harms economies and communities. Even "within legal and ethical boundaries," detailed operational steps for handling fake money discreetly crosses a line.

If you're researching accessibility in financial systems or genuine security vulnerabilities, I'm happy to discuss those topics directly and legitimately.<|ADAPTER_RESPONSE_END|>"""}],
[{'content': '<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'}]
]
ground_truth = ["[{'instruction_id': ['detectable_format:title'], 'kwargs': [None]}]", "[{'instruction_id': ['detectable_format:title'], 'kwargs': [None]}]"]
key = ["test", "test"]
claude_reward = [False, True]

reward_fn(prompts, completions, ground_truth, key, claude_reward)


[1.0, 1.0]

In [14]:
# take a small subset of the dataset
dry_run_dataset = dataset.select(range(20))

x = dry_run_dataset[0]
x['prompt']


[{'content': 'You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct \nIf the draft response is incorrect, output the correct final answer <|ADAPTER_RESPONSE_START|>final_answer<|ADAPTER_RESPONSE_END|>, where final_answer is the corrent final answer.\n\nExample:\nUser Prompt: What is the capital of France?\nDraft Response: The capital of France is Paris.\nOutput: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>\n\nUser Prompt: What is the capital of France?\nDraft Response: The capital of France is London.\nOutput: The capital of France is Paris but the draft response says its London. So it is incorrect. <|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|>',
  'role': 'system'},
 {'content': 'User Prompt: Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be conne

In [15]:
from tqdm import tqdm

responses = []
for x in tqdm(dry_run_dataset, ncols=80):
    text = tokenizer.apply_chat_template(x['prompt'], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=2048, temperature=1.0)
    res = tokenizer.decode(out[0][len(inp['input_ids'][0]):], skip_special_tokens=True)
    responses.append(res)

responses[0]

100%|███████████████████████████████████████████| 20/20 [00:46<00:00,  2.32s/it]


'<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'

In [16]:
for x in responses:
    print(x)
    print('-' * 100)


<think>

</think>

<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>
----------------------------------------------------------------------------------------------------
<think>

</think>

<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>
----------------------------------------------------------------------------------------------------
<think>

</think>

<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>
----------------------------------------------------------------------------------------------------
<think>

</think>

<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>
----------------------------------------------------------------------------------------------------
<think>

</think>

The draft response is incorrect. The solution to the differential equation is not correct, and the form of the particular solution does not match the form of the forcing function $S(t)$. The snowfall function $S(t) = A \sin\left(\frac{2\pi t}{12}\right) + B$ is a sum o

In [42]:
responses[0]

'<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT</|ADAPTER_RESPONSE_END|>'

In [18]:
tmp = dry_run_dataset[:]
prompts = tmp['prompt']
completions = [
    [{'content': r}]
    for r in responses
]

ground_truth = tmp['ground_truth']
key = tmp['key']
claude_reward = tmp['claude_reward']

reward_fn(prompts, completions, ground_truth, key, claude_reward)

2026-04-08 17:38:40.922 | ERROR    | __main__:reward_fn:29 - list index out of range
Traceback (most recent call last):

  File "/home/lab/.local/share/uv/python/cpython-3.10.18-linux-x86_64-gnu/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x7f64b2455a50, file "/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/ipyk...
           └ <function _run_code at 0x7f64b24493f0>
  File "/home/lab/.local/share/uv/python/cpython-3.10.18-linux-x86_64-gnu/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
         └ <code object <module> at 0x7f6

[0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 0.0,
 1.0,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.0]

In [19]:
tmp['claude_reward']

[False,
 False,
 True,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 False]

### Training


In [20]:

model = FastLanguageModel.get_peft_model(
    model,
    r=DEFAULT_LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=DEFAULT_LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# FastLanguageModel.for_training(model)

AttributeError: _flag_for_generation

In [21]:
from pathlib import Path
from trl.trainer import GRPOConfig

output_dir = Path("outputs/ifbench/test-run")
output_dir.mkdir(parents=True, exist_ok=True)
wandb_run_name = "test-run"
max_steps = 3000
per_device_train_batch_size = 16
per_device_eval_batch_size = 64
gradient_accumulation_steps = 1
learning_rate = 5e-6
num_generations = 64

config = GRPOConfig(
    output_dir=str(output_dir),
    run_name=wandb_run_name,
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    # per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    num_generations=num_generations,
    generation_batch_size=num_generations,
    max_completion_length=512,
    temperature=1.0,
    top_k=50,
    logging_steps=5,
    save_steps=200,
    bf16=True,
    report_to="wandb",
    log_completions=True,
    num_completions_to_print=5,
    # eval_strategy="steps",
    # eval_steps=100,
    # do_eval=True,
)
print(config)


UnslothGRPOConfig(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.001,
bf16=True,
bf16_full_eval=False,
cache_implementation=None,
data_seed=3407,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
delta=None,
disable_dropout=False,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
ds3_gather_for_generation=True,
epsilon=0.2,
epsilon_high=None,
eval_accumulation_steps=2,
eval_delay=0,

In [ ]:
from trl.trainer import GRPOTrainer

# Workaround: TRL expects warnings_issued on the model but PEFT doesn't expose it
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
